# NB07 — Diagnostic Plots

> **The four standard plots every regression analysis should include.**

1. Residuals vs Fitted
2. Normal Q-Q
3. Scale-Location (spread-location)
4. Residuals vs Leverage (Cook's distance)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

np.random.seed(42)
n = 60
X_raw = np.linspace(1, 10, n)
y = 2.5*X_raw + 4 + np.random.normal(0, 3, n)

# Inject one outlier
y[50] = y[50] + 30

Xsm = sm.add_constant(X_raw)
result = sm.OLS(y, Xsm).fit()
influence = result.get_influence()

fitted       = result.fittedvalues
resid        = result.resid
std_resid    = influence.resid_studentized_internal
leverage     = influence.hat_matrix_diag
cooks_d      = influence.cooks_distance[0]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, resid, s=25, alpha=0.6, color='steelblue')
ax.axhline(0, color='red', linewidth=1.2)
# Lowess smoother
from statsmodels.nonparametric.smoothers_lowess import lowess
lo = lowess(resid, fitted, frac=0.4)
ax.plot(lo[:,0], lo[:,1], 'r-', linewidth=2, label='Lowess')
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
ax.set_title('1. Residuals vs Fitted')
ax.legend(); ax.grid(alpha=0.3)

# 2. Normal Q-Q
ax = axes[0, 1]
(osm, osr), (slope, intercept, _) = stats.probplot(std_resid)
ax.scatter(osm, osr, s=25, alpha=0.6, color='steelblue')
xs = np.array([osm.min(), osm.max()])
ax.plot(xs, slope*xs+intercept, 'r-', linewidth=2)
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Standardised residuals')
ax.set_title('2. Normal Q-Q')
ax.grid(alpha=0.3)

# 3. Scale-Location
ax = axes[1, 0]
sqrt_std_resid = np.sqrt(np.abs(std_resid))
ax.scatter(fitted, sqrt_std_resid, s=25, alpha=0.6, color='steelblue')
lo2 = lowess(sqrt_std_resid, fitted, frac=0.4)
ax.plot(lo2[:,0], lo2[:,1], 'r-', linewidth=2)
ax.set_xlabel('Fitted values'); ax.set_ylabel('√|Std. residuals|')
ax.set_title('3. Scale-Location')
ax.grid(alpha=0.3)

# 4. Residuals vs Leverage (Cook's D)
ax = axes[1, 1]
ax.scatter(leverage, std_resid, s=25, alpha=0.6, color='steelblue')
ax.axhline(0, color='red', linewidth=0.8)
for i in np.argsort(cooks_d)[-5:]:
    ax.annotate(f'obs {i}', (leverage[i], std_resid[i]),
                textcoords='offset points', xytext=(5,5), fontsize=8)
threshold = 4 / n
ax.axvline(threshold, color='orange', linestyle='--', linewidth=1.2, label=f'Leverage = 4/n')
ax.set_xlabel('Leverage'); ax.set_ylabel('Std. residuals')
ax.set_title("4. Residuals vs Leverage (Cook's D)")
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Regression Diagnostic Plots', fontsize=14)
plt.tight_layout(); plt.show()


## How to read each plot

### 1. Residuals vs Fitted
- **Good:** random scatter around y=0, no pattern
- **Curved pattern:** linearity assumption violated
- **Funnel shape:** heteroscedasticity

### 2. Normal Q-Q
- **Good:** points lie on the diagonal line
- **Heavy tails (S-curve):** residuals have heavier tails than normal
- **Skew:** points systematically above or below the line

### 3. Scale-Location
- **Good:** flat horizontal band of points
- **Rising/widening band:** heteroscedasticity (variance increases with fitted value)

### 4. Residuals vs Leverage (Cook's Distance)
- **Leverage:** how unusual is point i in X-space?
- **Influence:** how much does point i change the coefficients if removed?
- **Cook's D > 4/n:** potentially influential observation


In [ ]:
# Cook's distance — identify influential points
import pandas as pd

top5 = np.argsort(cooks_d)[-5:][::-1]
df_inf = pd.DataFrame({
    'obs': top5,
    "Cook's D": cooks_d[top5].round(4),
    'Leverage': leverage[top5].round(4),
    'Std resid': std_resid[top5].round(3),
})
print("Top 5 influential observations:")
print(df_inf.to_string(index=False))
print(f"\nThreshold Cook's D = 4/n = {4/n:.4f}")
print(f"Observation 50 was our injected outlier.")


## Key Takeaways

- Run all 4 diagnostic plots **every time** you fit a regression
- Residuals vs Fitted + Scale-Location: check L and E assumptions
- Q-Q plot: check N assumption
- Leverage plot: find influential outliers that distort the fit

**Next:** NB08 — Multiple Linear Regression (matrix form, multiple predictors)
